Projekt przedstawia zastosowanie uczenia nienadzorowanego do analizy kampanii marketingowych. Za pomocą algorytmu K-Means kampanie zostały pogrupowane według podobieństwa wybranych wskaźników efektywności, co pozwoliło wyróżnić różne typy kampanii i lepiej zrozumieć ich charakterystykę.

Dane wykorzystane w repozytorium są syntetyczne i zostały wygenerowane na potrzeby projektu, tak aby odzwierciedlały przykładowe wyniki kampanii marketingowych bez użycia rzeczywistych danych firmowych.

In [2]:
# pip install plotly
# pip install seaborn
# pip install tensorflow
# pip install nbformat

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dropout, Dense
from tensorflow.keras.activations import elu, exponential, hard_sigmoid, linear, relu, sigmoid, softmax, tanh
from tensorflow.keras.losses import binary_crossentropy, categorical_crossentropy, logcosh, MeanSquaredError, poisson, MeanAbsoluteError
from tensorflow.keras.metrics import AUC
from tensorflow.keras.optimizers import Adadelta, Adam, Nadam, RMSprop, SGD, Ftrl
from sklearn.model_selection import train_test_split
from sklearn import datasets
from keras.datasets import reuters
from sklearn.linear_model import Perceptron
from tensorflow.keras.preprocessing.text import Tokenizer #dot. analizy tekstu
import keras
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [2]:
data = pd.read_excel("wyniki_kampanii_syntetyczne.xlsx",header=2)

In [3]:
data

,Kampania,Wyświetlenia,Kliknięcia,CTR,Śr. CPC,Koszt,Konwersje,Wartość konw.,ROAS
0,Campaign_1,4231,114,0.027146,6.14,699.96,4,888.19,1.27
1,Campaign_2,3300,118,0.035951,4.65,548.70,6,223.18,0.41
2,Campaign_3,9651,315,0.032640,3.00,945.00,12,304.71,0.32
3,Campaign_4,29391,1972,0.067113,6.93,13665.96,102,2708.08,0.20
4,Campaign_5,48774,2020,0.041417,7.12,14382.40,41,2394.86,0.17
...,...,...,...,...,...,...,...,...,...
115,Campaign_116,36925,1619,0.043873,3.28,5310.32,160,21499.00,4.05
116,Campaign_117,43260,2789,0.064471,6.60,18407.40,262,67978.50,3.69
117,Campaign_118,43546,2965,0.068102,3.04,9013.60,85,9613.51,1.07
118,Campaign_119,39199,535,0.013649,6.38,3413.30,50,9043.75,2.65


In [4]:
data_clean = data.dropna()
data_clean.head()


,Kampania,Wyświetlenia,Kliknięcia,CTR,Śr. CPC,Koszt,Konwersje,Wartość konw.,ROAS
0,Campaign_1,4231,114,0.027146,6.14,699.96,4,888.19,1.27
1,Campaign_2,3300,118,0.035951,4.65,548.70,6,223.18,0.41
2,Campaign_3,9651,315,0.032640,3.00,945.00,12,304.71,0.32
3,Campaign_4,29391,1972,0.067113,6.93,13665.96,102,2708.08,0.20
4,Campaign_5,48774,2020,0.041417,7.12,14382.40,41,2394.86,0.17


In [5]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# wybór cech
X = data_clean[['Kliknięcia', 'Wyświetlenia', 'Śr. CPC', 'Koszt', 'Konwersje', 'Wartość konw.']]

# skalowanie
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# klastrowanie
kmeans = KMeans(n_clusters=3, random_state=123, n_init=10)
data_clean['Cluster'] = kmeans.fit_predict(X_scaled)

# podgląd
print(data_clean[['Kampania', 'Cluster']])

         Kampania  Cluster
0      Campaign_1        2
1      Campaign_2        2
2      Campaign_3        2
3      Campaign_4        1
4      Campaign_5        1
..            ...      ...
115  Campaign_116        1
116  Campaign_117        0
117  Campaign_118        1
118  Campaign_119        1
119  Campaign_120        2

[120 rows x 2 columns]


In [7]:
data_clean[data_clean['Koszt']>0][["Kampania","Kliknięcia", "Wyświetlenia","Koszt", "Konwersje","Wartość konw.","ROAS","Cluster"]]

,Kampania,Kliknięcia,Wyświetlenia,Koszt,Konwersje,Wartość konw.,ROAS,Cluster
0,Campaign_1,114,4231,699.96,4,888.19,1.27,2
1,Campaign_2,118,3300,548.70,6,223.18,0.41,2
2,Campaign_3,315,9651,945.00,12,304.71,0.32,2
3,Campaign_4,1972,29391,13665.96,102,2708.08,0.20,1
4,Campaign_5,2020,48774,14382.40,41,2394.86,0.17,1
...,...,...,...,...,...,...,...,...
115,Campaign_116,1619,36925,5310.32,160,21499.00,4.05,1
116,Campaign_117,2789,43260,18407.40,262,67978.50,3.69,0
117,Campaign_118,2965,43546,9013.60,85,9613.51,1.07,1
118,Campaign_119,535,39199,3413.30,50,9043.75,2.65,1
